In [ ]:
!pip install ortools

In [ ]:
from ortools.linear_solver import pywraplp

In [ ]:
tamanho_barra = int(input("Informe o tamanho da barra original (ex: 150): "))
qtd_tipos = int(input("Informe a quantidade de tipos de itens da demanda (ex: 3): "))

tamanhos_itens = []
demandas_itens = []

for i in range(qtd_tipos):
    tam = int(input(f"Tamanho do item {i+1}: "))
    dem = int(input(f"Demanda do item {i+1}: "))
    tamanhos_itens.append(tam)
    demandas_itens.append(dem)

In [ ]:
print("\n" + "="*40)
print("GERANDO PADRÕES DE CORTE...")
print("="*40)

padroes = []
desperdicios = []


menor_peca = min(tamanhos_itens)

def gerar_padroes(idx_item, padrao_atual, tamanho_ocupado):

    if idx_item == qtd_tipos:
        desperdicio_atual = tamanho_barra - tamanho_ocupado


        if sum(padrao_atual) > 0 and desperdicio_atual < menor_peca:
            padroes.append(padrao_atual)
            desperdicios.append(desperdicio_atual)
        return

    max_qtd_possivel = (tamanho_barra - tamanho_ocupado) // tamanhos_itens[idx_item]

    for qtd in range(max_qtd_possivel + 1):
        novo_tamanho = tamanho_ocupado + (qtd * tamanhos_itens[idx_item])
        gerar_padroes(idx_item + 1, padrao_atual + [qtd], novo_tamanho)

gerar_padroes(0, [], 0)

print("\n" + "="*70)
print("PADRÕES DE CORTE GERADOS")
print("="*70)

print(f"{'Padrão':<10} {'Composição do corte':<30} {'Vetor':<15} {'Sobra'}")
print("-"*70)

for i, p in enumerate(padroes):

    descricao = []

    for j in range(qtd_tipos):
        if p[j] > 0:
            descricao.append(f"{p[j]}x{tamanhos_itens[j]}")

    descricao_final = " + ".join(descricao)

    print(
        f"{i+1:<10} "
        f"{descricao_final:<30} "
        f"{str(p):<15} "
        f"{desperdicios[i]}"
    )

In [ ]:
solver = pywraplp.Solver.CreateSolver('SCIP')
infinity = solver.infinity()

x = {}
for i in range(len(padroes)):
    x[i] = solver.IntVar(0, infinity, f"x_{i+1}")

# Função objetivo:
# minimizar a quantidade total de barras usadas
objetivo = solver.Objective()
for i in range(len(padroes)):
    objetivo.SetCoefficient(x[i], 1)

objetivo.SetMinimization()

# Restrições de demanda
for j in range(qtd_tipos):
    restricao = solver.Constraint(demandas_itens[j], infinity, f"Demanda_Tamanho_{tamanhos_itens[j]}")
    for i in range(len(padroes)):
        restricao.SetCoefficient(x[i], padroes[i][j])

In [ ]:
print("\n" + "="*70)
print("MODELO DE PROGRAMAÇÃO LINEAR INTEIRA")
print("="*70)
print(solver.ExportModelAsLpFormat(False))

print("\n" + "="*70)
print("RESULTADO DA OTIMIZAÇÃO")
print("="*70)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:

    print("SOLUÇÃO ÓTIMA ENCONTRADA!\n")
    print(f"Valor da função objetivo: {int(objetivo.Value())}")

    print("\n" + "="*100)
    print("QUANTIDADE A CORTAR DE CADA PADRÃO")
    print("="*100)

    total_barras_usadas = 0
    desperdicio_total_real = 0

    print(
        f"{'Padrão':<10} "
        f"{'Corte de cada barra':<35} "
        f"{'Qtd. a cortar':<15} "
        f"{'Desperdício unit.':<20} "
        f"{'Desperdício total'}"
    )
    print("-"*100)

    for i in range(len(padroes)):

        qtd_usada = int(x[i].solution_value())

        if qtd_usada > 0:

            total_barras_usadas += qtd_usada
            desperdicio_padrao = qtd_usada * desperdicios[i]
            desperdicio_total_real += desperdicio_padrao

            descricao = []

            for j in range(qtd_tipos):
                if padroes[i][j] > 0:
                    descricao.append(f"{padroes[i][j]} peça(s) de {tamanhos_itens[j]}m")

            descricao_final = " + ".join(descricao)

            print(
                f"{i+1:<10} "
                f"{descricao_final:<35} "
                f"{qtd_usada:<15} "
                f"{desperdicios[i]:<20} "
                f"{desperdicio_padrao}"
            )

    print("\n" + "="*100)
    print(f"TOTAL DE BARRAS ORIGINAIS UTILIZADAS : {total_barras_usadas}")
    print(f"DESPERDÍCIO TOTAL FINAL              : {desperdicio_total_real}")
    print("="*100)